In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
from statistics import mean
import joblib

from lime import lime_tabular

In [3]:
# Remove unnecessary columns
def clean_csv(old_filename, new_filename, remove_col_list):
    try:
        df = pd.read_csv(old_filename)
        df.drop(columns=remove_col_list, inplace=True)

        # for i, row in enumerate(df):
        #     df.iloc[i, 1] = len(df["URL"].iloc[i])

        df.to_csv(new_filename, index=False)

        print(f'file: "{old_filename}" cleaned successfully')
        return True
    except Exception as e:
        print(f'file: "{old_filename}" clean failed ({e})')
        return False

In [4]:
# Loads features and target variables
def load_data(filename, y_target_name, num_rows=-1, pos=0):
    df = pd.read_csv(filename)
    if num_rows > 0:
        df = df[pos : pos + num_rows]
    elif pos != 0:
        df = df[pos:]

    X = df.drop(columns=[y_target_name])
    y = df[y_target_name]

    return X, y, df

In [5]:
def show_corr_matrix(df, output_filename):
    non_valid_cols = [
        df.columns[i]
        for i, item in enumerate(df.dtypes)
        if not pd.api.types.is_numeric_dtype(item)
    ]
    df.drop(columns=non_valid_cols, inplace=True)
    matrix = df.corr()
    plt.figure(figsize=(32, 16))
    sns.heatmap(matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title("UCI Phishing URL Correlation Heatmap")
    plt.savefig(output_filename)

In [6]:
def get_training_test_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
    return X_train, X_test, y_train, y_test

In [7]:
def logistic_regression_model_training(X_train, X_test, y_train, y_test):
    C_grid = np.logspace(-4, 4, 20)
    y_train_log_loss_data = []
    y_test_log_loss_data = []
    for i, C in enumerate(C_grid):
        print(f"working on {i} with C value: {C:.4f}")
        clf = LogisticRegression(C=C, l1_ratio=0, solver="lbfgs", max_iter=500).fit(
            X_train, y_train
        )
        y_train_pred = clf.predict_proba(X_train)
        y_test_pred = clf.predict_proba(X_test)
        y_train_log_loss = log_loss(y_train, y_train_pred)
        y_test_log_loss = log_loss(y_test, y_test_pred)

        # print(f"{y_train_pred=}")
        # print(f"{y_test_pred=}")
        print(f"{y_train_log_loss=:.4f}")
        print(f"{y_test_log_loss=:.4f}")
        y_train_log_loss_data.append(y_train_log_loss)
        y_test_log_loss_data.append(y_test_log_loss)

    plt.figure()
    plt.plot(
        C_grid,
        y_train_log_loss_data,
        "o-",
        color="blue",
        label="Training Data Log-Loss",
        markersize=3,
    )
    plt.plot(
        C_grid,
        y_test_log_loss_data,
        "o-",
        color="red",
        label="Test Data Log-Loss",
        markersize=3,
    )

    plt.xlabel("C")
    plt.ylabel("Log-Loss")
    plt.title("Log-Loss of Training and Test Data Vs C")
    # plt.ylim([0, 1])
    plt.xscale("log")
    plt.legend()
    plt.savefig("../outputs/logistic_regression_log_loss_c.png")

In [8]:
# clean_csv(
#     "../data/uci_phishing_url_dataset.csv",
#     "../data/uci_phishing_url_dataset_clean.csv",
#     [
#         "URL",
#         "Domain",
#         "TLD",
#         "Title",
#         "HasTitle",
#         "DomainTitleMatchScore",
#         "URLTitleMatchScore",
#         "HasFavicon",
#         "Robots",
#         "IsResponsive",
#         "NoOfURLRedirect",
#         "NoOfSelfRedirect",
#         "HasDescription",
#         "NoOfPopup",
#         "NoOfiFrame",
#         "HasExternalFormSubmit",
#         "HasSocialNet",
#         "HasSubmitButton",
#         "HasHiddenFields",
#         "HasPasswordField",
#         "Bank",
#         "Pay",
#         "Crypto",
#         "HasCopyrightInfo",
#         "NoOfImage",
#         "NoOfCSS",
#         "NoOfJS",
#         "NoOfSelfRef",
#         "NoOfEmptyRef",
#         "NoOfExternalRef",
#         "URLSimilarityIndex",
#         "CharContinuationRate",
#         "TLDLegitimateProb",
#         "URLCharProb",
#         "LineOfCode",
#         "LargestLineLength",
#     ],
# )

remove "FILENAME" column from data


In [9]:
X, y, df = load_data("../data/uci_phishing_url_dataset_new.csv", "IsLegit", 100000)

In [10]:
# show_corr_matrix(df, "../outputs/uci_phishing_url_corr_matrix.png")

In [11]:
X_train, X_test, y_train, y_test =  get_training_test_data(X.drop(columns="URL"), y)
# logistic_regression_model_training(X_train, X_test, y_train, y_test)

In [118]:
clf = LogisticRegression(C=5, solver="lbfgs", max_iter=500).fit(X_train, y_train)
explainer_lime = lime_tabular.LimeTabularExplainer(X_train.values, feature_names=saved_features,  mode="regression", random_state=0)

joblib.dump({"features": X_train.columns.tolist(), "model": clf}, "../models/logit_model.joblib")
joblib.dump(X_train, "../models/lime_training_data.joblib")

['../models/lime_training_data.joblib']

In [ ]:
model_dump = joblib.load("../models/logit_model.joblib")
saved_features = model_dump["features"]
saved_clf = model_dump["model"]
X_new, y_new, df_new = load_data("../data/uci_phishing_url_dataset_new.csv", "IsLegit", pos=100000)
print(saved_clf.coef_)
y_new_pred = saved_clf.predict(X_new[saved_features])

accuracy = accuracy_score(y_new, y_new_pred)
print(f"Accuracy: {accuracy}")

[[-4.43795575e+00  8.26612659e+00 -8.63420920e-03  4.37987686e+00
   5.64396371e-01  2.97032724e+00 -6.16798430e-01 -4.62463988e-01
  -2.72564122e-03 -1.53507562e-02 -1.92503587e-04 -3.79663015e+00
  -7.29660586e-01 -4.41318443e+00 -2.05779776e-01 -3.94933727e-02
  -4.12531290e-02 -5.11944395e-03 -4.66751693e+00 -1.07823227e-01
   1.74057230e+01]]
Accuracy: 0.9976508707978938


# XAI using LIME


In [112]:
explainer_lime = model_dump["explainer"]

In [115]:
NUM_TOP_FEATURES = 10
idx = 0

data_point = X.iloc[idx]
url = data_point["URL"]
url_info = data_point.drop("URL").to_numpy()
print(url)

exp_lime = explainer_lime.explain_instance(url_info, lambda x: saved_clf.predict_proba(pd.DataFrame(x, columns=saved_features)), num_features=NUM_TOP_FEATURES)
explanations = exp_lime.as_list()
for explanation in explanations:
    print(explanation)

https://www.southbankmosaics.com
('ObfuscationRatio <= 0.00', 0.27205851269149955)
('NoOfDigitsInURL <= 0.00', -0.1608664847956361)
('HasObfuscation <= 0.00', 0.13720668566785083)
('20.00 < DomainLength <= 24.00', -0.11960566533733281)
('NoOfQMarkInURL <= 0.00', -0.09560429862429197)
('IsDomainIP <= 0.00', -0.06627288495577291)
('NoOfOtherSpecialCharsInURL <= 1.00', -0.04748327458112282)
('NoOfAmpersandInURL <= 0.00', 0.04339371203157486)
('14.00 < NoOfLettersInURL <= 20.00', 0.024253865167650435)
('SpecialCharRatioInURL <= 0.04', 0.01501042919308461)
